# Stage 2 retrieval ablation — Jupyter runner

This notebook runs the same pipeline as the `.py` files in `src/stage2/`
(`build_index.py`, `retrieval.py`, `translate.py`, `run_ablations.py`) —
just without needing the command line. Run the cells top to bottom.

**Order:**
1. Setup (point this notebook at your repo, install deps)
2. Build the FAISS retrieval index(es)
3. Sanity-check with a free dry run (no API calls)
4. Configure and run the k×query-arm sweep
5. Score everything and view the headline result

The headline result (Step 5) is **Table 2**: does culturally-indexed
retrieval beat vanilla text retrieval, holding the prompt mode and k fixed?
That's the actual RQ1 test — everything before it just gets you there.


## Step 0 — Setup

In [ ]:
import sys
from pathlib import Path

# EDIT THIS to the folder that contains "src/" (i.e. your repo root).
REPO_ROOT = Path("/path/to/your/repo")

assert (REPO_ROOT / "src" / "stage2").exists(), (
    f"Can't find src/stage2 under {REPO_ROOT} -- fix REPO_ROOT above and re-run this cell."
)
sys.path.insert(0, str(REPO_ROOT))
print(f"Repo root OK: {REPO_ROOT}")


In [ ]:
# One-time installs. Safe to re-run; already-installed packages are skipped.
%pip install -q faiss-cpu sentence-transformers google-genai pandas


### Vertex AI auth (only needed if you plan to actually call Gemini, i.e. `DRY_RUN = False` below)

Run these **in a terminal**, not in this notebook (they open a browser login):

```
gcloud auth application-default login
gcloud auth application-default set-quota-project cicil-501318
gcloud services enable aiplatform.googleapis.com --project cicil-501318
```

The cell below just checks whether that's already done.


In [ ]:
import os

adc = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
if os.path.exists(adc) or os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
    print("Vertex AI credentials found -- real (non-dry-run) translation will work.")
else:
    print("No Vertex AI credentials found yet.")
    print("Either run the gcloud commands above in a terminal, or keep DRY_RUN = True below.")


## Step 1 — Build the retrieval index(es)

Reads whatever's in `build_index.CORPORA` today (just the 20-pair Wixárika
pilot, until more banks land) and writes a FAISS index per language to
`indices/`. Safe to re-run any time (e.g. after adding a new language to
`CORPORA`) -- it just rebuilds.


In [ ]:
from src.stage2 import build_index

build_index.main()


## Step 2 — Sanity check (dry run, no API calls, no cost)

Validates that records load, retrieval runs, and prompts build correctly for
the one language with a real index today (Wixárika). Writes sample prompts
to `predictions/_dryrun/` instead of calling Gemini.


In [ ]:
from src.stage2.translate import translate_language

for mode, query_arm in [("generic", "text"), ("cultural-vqa", "cultural"), ("cultural-vqa", "text")]:
    translate_language("wixarika", mode, k=5, query_arm=query_arm, dry_run=True)
    print()

print("Sample prompts written to predictions/_dryrun/ -- inspect one below.")


In [ ]:
from src.stage2.paths import PRED_DIR

sample = PRED_DIR / "_dryrun" / "wixarika_cultural-vqa_culturalquery_k5_prompt_sample.txt"
print(sample.read_text(encoding="utf-8")[:2500])


## Step 3 — Configure the sweep

Edit the variables below, then run Step 4.

- `LANGUAGES`: languages to run. Ones without a built index (see
  `build_index.CORPORA`) still run -- they just fall back to zero-shot
  translation, same as before.
- `RUN_CONFIGS`: (prompt mode, retrieval query-arm) pairs. `generic+cultural`
  is deliberately not included -- generic-mode records carry no
  `cultural_annotations`, so it would just duplicate `generic+text`.
- `DRY_RUN = True`: validates the whole grid with **zero Gemini calls, zero
  cost**. Flip to `False` only once you're ready to actually spend quota
  (and have Vertex auth set up above).


In [ ]:
LANGUAGES = ["guarani", "bribri", "maya", "wixarika", "nahuatl"]

K_VALUES = [3, 5, 8]

RUN_CONFIGS = [
    ("generic", "text"),
    ("cultural-vqa", "cultural"),
    ("cultural-vqa", "text"),
]

DRY_RUN = True   # <-- flip to False to actually call Gemini


## Step 4 — Run the sweep

This is the notebook equivalent of `run_sweep.py`. With `DRY_RUN = True`
it's free and just re-validates every combination. With `DRY_RUN = False` it
calls Gemini once per record per run -- for 5 languages x 3 arms x 3 k-values
that's 45 runs total, so expect it to take a while and use real quota.

A failure in one run (e.g. a language whose input JSONL doesn't exist yet)
is caught and skipped so it doesn't stop the rest of the grid.


In [ ]:
import time
from src.stage2.translate import ensure_vertex_credentials

if not DRY_RUN:
    ensure_vertex_credentials()

total = len(LANGUAGES) * len(RUN_CONFIGS) * len(K_VALUES)
done = 0

for lang in LANGUAGES:
    for mode, query_arm in RUN_CONFIGS:
        for k in K_VALUES:
            done += 1
            print(f"[{done}/{total}] {lang} | mode={mode} | query-arm={query_arm} | k={k}")
            try:
                translate_language(lang, mode, k, dry_run=DRY_RUN, query_arm=query_arm)
            except Exception as e:
                print(f"  SKIPPED ({lang}/{mode}/{query_arm}/k={k}): {e}")
            if not DRY_RUN:
                time.sleep(1.0)

print("\nDry-run sweep complete." if DRY_RUN else "\nSweep complete.")


## Step 5 — Score everything & view the headline result

This runs the same tables `run_ablations.py` prints on the command line.
**Table 2** is the headline RQ1 result: cultural-query vs text-query
retrieval, at k=5, with prompt mode held fixed at cultural-vqa.

(If you only ran the dry run in Step 4, there are no real prediction files
yet, so this will show `--` everywhere -- that's expected, not a bug.)


In [ ]:
from src.stage2 import run_ablations

run_ablations.main()


### Same headline table, as a DataFrame (easier to read/plot/export)

In [ ]:
import pandas as pd
from src.stage1.evaluate import score_translations
from src.stage2.run_ablations import pred_path, LANGUAGES as ABLATION_LANGS

rows = []
for lang in ABLATION_LANGS:
    cultural_file = pred_path(lang, "cultural-vqa", 5, query_arm="cultural")
    text_file = pred_path(lang, "cultural-vqa", 5, query_arm="text")

    cultural_score = text_score = None
    if cultural_file.exists():
        cultural_score, _ = score_translations(lang, cultural_file, split="dev")
    if text_file.exists():
        text_score, _ = score_translations(lang, text_file, split="dev")

    delta = (cultural_score - text_score) if (cultural_score is not None and text_score is not None) else None
    rows.append({
        "language": lang,
        "cultural_query_k5": cultural_score,
        "text_query_k5": text_score,
        "delta": delta,
    })

df = pd.DataFrame(rows)
df


## Troubleshooting

- **`FileNotFoundError: Index not found`** -- Step 1 (build_index) hasn't
  been run for that language, or `CORPORA` in `build_index.py` doesn't have
  an entry for it yet. That language falls back to zero-shot automatically,
  it won't crash the sweep.
- **`WARNING: ... not found -- skipping`** during Step 1 -- the Stage 1
  input file for that language/split doesn't exist yet. Fine, just means no
  index for that language until it does.
- **ADC / Vertex auth errors during Step 4** -- run the `gcloud` commands
  from Step 0 in a terminal, not in this notebook.
- **Table shows `--` everywhere in Step 5** -- either you only dry-ran the
  sweep (Step 4 `DRY_RUN=True`), or the prediction files are named
  differently than `run_ablations.py` expects:
  `{lang}_{mode}[_{backend}][_{query_arm}query]_k{k}_predictions.txt` (the
  `{backend}` segment is only present when translate.py was run with
  `--backend` set to something other than `ollama`, e.g. `smolvlm`).
